# Màquines de vectors de suport (SVM)

**Optativa d'Aprenentatge automàtic · DAM/DAW 2n**

Aquest és l'últim quadern de la sèrie. Hem vist quatre maneres de separar
dades en grups: distàncies (k-NN), condicions en cadena (arbre), un
comitè d'arbres votant (bosc) i una frontera amb probabilitat (regressió
logística).

La regressió logística ens deixava una pregunta oberta: si les dades es
poden separar amb una recta, **n'hi ha moltes, de rectes que ho fan bé**.
La logística en tria una seguint la probabilitat, però no diu que sigui
la millor situada. Avui contestem això: **quina és la millor recta, i
per què**.

## 1. Tres rectes, totes perfectes

Agafem dues espècies d'Iris que se separen molt bé si mirem només la
llargada i l'amplada del pètal: *setosa* i *versicolor*.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris

iris = load_iris(as_frame=True)
dades = iris.frame.copy()
dades["especie"] = iris.target_names[iris.target]
dades = dades.rename(columns={
    "petal length (cm)": "petal_llarg",
    "petal width (cm)": "petal_ample",
})

dos = dades[dades["especie"].isin(["setosa", "versicolor"])].reset_index(drop=True)
X = dos[["petal_llarg", "petal_ample"]].to_numpy()
y = (dos["especie"] == "versicolor").to_numpy().astype(int)

print(f"Mostres: {len(X)} ({(y==0).sum()} setosa, {(y==1).sum()} versicolor)")
dos.groupby("especie")[["petal_llarg", "petal_ample"]].agg(["min", "max"])

Hi ha un buit clar entre les dues espècies: la *setosa* no passa mai
d'un pètal de 1,9 cm de llarg, i la *versicolor* no baixa mai de 3,0 cm.
Qualsevol recta que passi per aquest buit separa les dues classes al
100 %.

Aquí en tens tres, totes diferents, totes amb un 100 % d'encerts a
l'entrenament:

- **Recta A**: vertical, a $x_1 = 2{,}2$.
- **Recta B**: vertical, a $x_1 = 2{,}5$.
- **Recta C**: diagonal, $x_1 + x_2 = 3{,}2$.

(Aquí $x_1$ és la llargada del pètal i $x_2$ l'amplada.)

In [ ]:
rectes = {
    "Recta A": {"w": np.array([1.0, 0.0]), "b": -2.2, "color": "tab:red"},
    "Recta B": {"w": np.array([1.0, 0.0]), "b": -2.5, "color": "tab:purple"},
    "Recta C": {"w": np.array([1.0, 1.0]), "b": -3.2, "color": "tab:green"},
}

plt.figure(figsize=(7, 6))
plt.scatter(X[y == 0, 0], X[y == 0, 1], label="setosa", color="tab:blue")
plt.scatter(X[y == 1, 0], X[y == 1, 1], label="versicolor", color="tab:orange")

x1_linia = np.linspace(0.5, 5.5, 100)
for nom, r in rectes.items():
    w1, w2 = r["w"]
    if w2 == 0:
        plt.axvline(-r["b"] / w1, color=r["color"], label=nom)
    else:
        x2_linia = (-r["b"] - w1 * x1_linia) / w2
        plt.plot(x1_linia, x2_linia, color=r["color"], label=nom)

plt.xlim(0.5, 5.5)
plt.ylim(-0.1, 2.0)
plt.xlabel("Llargada del pètal (cm)")
plt.ylabel("Amplada del pètal (cm)")
plt.title("Tres rectes, les tres amb 100% d'encerts")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

Les tres encerten totes les flors d'entrenament. Cap és "més correcta"
que les altres si només mirem la precisió. Però imagina que arriba una
flor nova, una mica desplaçada per l'atzar de la mesura: **amb quina
recta tens més marge d'error abans d'equivocar-te?**

La resposta de l'SVM: tria **la recta que passa més lluny de tots els
punts**. Com més marge, més tolerància a que una flor nova surti una
mica descol·locada.

## 2. El marge, amb geometria de batxillerat

Una recta en aquest pla es pot escriure com

$$w_1 x_1 + w_2 x_2 + b = 0$$

i la distància d'un punt $(x_1, x_2)$ a aquesta recta és

$$d = \frac{|w_1 x_1 + w_2 x_2 + b|}{\sqrt{w_1^2 + w_2^2}}$$

El denominador és Pitàgoras un altre cop: $\sqrt{w_1^2+w_2^2}$ és la
longitud del vector $(w_1, w_2)$, que és perpendicular a la recta.

El **marge** d'una recta és la distància mínima entre la recta i el
punt més proper, de qualsevol de les dues classes. Calculem-lo per a
les tres rectes de dalt.

In [ ]:
def distancia(w, b, X):
    return np.abs(X @ w + b) / np.linalg.norm(w)

for nom, r in rectes.items():
    d = distancia(r["w"], r["b"], X)
    marge_setosa = d[y == 0].min()
    marge_versicolor = d[y == 1].min()
    marge = d.min()
    print(f"{nom}: marge setosa={marge_setosa:.3f}  marge versicolor={marge_versicolor:.3f}"
          f"  ->  marge total={marge:.3f}")

**Recta C guanya**, i no perquè ho digui jo: ho diu el càlcul. És la que
queda més lluny de la flor de *setosa* i de la de *versicolor* que
tenia més a prop.

Ara imagina que en comptes de tres rectes en proves totes les possibles
i et quedes amb la de marge més gran. Això és exactament un problema
d'**optimització**: trobar els $w_1, w_2, b$ que fan màxim el marge.
Nosaltres no el resoldrem a mà; és el que fa `SVC` per dins quan li
demanes `kernel="linear"`.

## 3. Els vectors de suport

Entrenem l'SVM de veritat i mirem quina recta troba.

In [ ]:
from sklearn.svm import SVC

model = SVC(kernel="linear", C=1e5, random_state=42)
model.fit(X, y)

w = model.coef_[0]
b = model.intercept_[0]
marge_svm = 1 / np.linalg.norm(w)
print(f"w = {w}, b = {b:.3f}")
print(f"Marge trobat per l'SVM: {marge_svm:.3f}")
print(f"Vectors de suport: {len(model.support_vectors_)} de {len(X)} mostres"
      f" ({len(model.support_vectors_)/len(X):.1%})")

El marge de l'SVM (calculat resolent l'optimització) surt **una mica
més gran** que el de la Recta C, que havíem triat a ull. Té sentit: la
Recta C era una bona aproximació, però l'SVM troba l'òptim exacte.

In [ ]:
def dibuixa_frontera_lineal(model, X, y, titol):
    plt.figure(figsize=(7, 6))
    plt.scatter(X[y == 0, 0], X[y == 0, 1], label="setosa", color="tab:blue")
    plt.scatter(X[y == 1, 0], X[y == 1, 1], label="versicolor", color="tab:orange")

    w = model.coef_[0]
    b = model.intercept_[0]
    x1_linia = np.linspace(0.5, 5.5, 100)
    x2_frontera = (-b - w[0] * x1_linia) / w[1]
    marge = 1 / np.linalg.norm(w)
    despl = marge * np.linalg.norm(w) / w[1]  # desplaçament vertical del marge

    plt.plot(x1_linia, x2_frontera, color="black", label="frontera")
    plt.plot(x1_linia, x2_frontera + despl, "k--", alpha=0.6, label="marge")
    plt.plot(x1_linia, x2_frontera - despl, "k--", alpha=0.6)

    plt.scatter(model.support_vectors_[:, 0], model.support_vectors_[:, 1],
                s=200, facecolors="none", edgecolors="red", linewidths=2,
                label="vectors de suport")

    plt.xlim(0.5, 5.5)
    plt.ylim(-0.1, 2.0)
    plt.xlabel("Llargada del pètal (cm)")
    plt.ylabel("Amplada del pètal (cm)")
    plt.title(titol)
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

dibuixa_frontera_lineal(model, X, y, "Frontera de l'SVM, marges i vectors de suport")

Fixa't en el nombre de vectors de suport: **només 2 de 100 mostres**.
Aquesta és la idea més important del quadern:

> **Només els punts marcats en vermell determinen on va la recta.**
> Tota la resta de flors podrien desaparèixer i la frontera no es
> mouria ni un mil·límetre.

Comprovem-ho de veritat: eliminem de l'entrenament unes quantes flors
que **no** són vectors de suport i tornem a entrenar.

In [ ]:
no_suport = [i for i in range(len(X)) if i not in set(model.support_)]
elimina = no_suport[:40]  # 40 flors que no aporten res a la frontera

mantenir = np.ones(len(X), dtype=bool)
mantenir[elimina] = False
X_retallat, y_retallat = X[mantenir], y[mantenir]

model_retallat = SVC(kernel="linear", C=1e5, random_state=42)
model_retallat.fit(X_retallat, y_retallat)

print(f"Entrenament original: {len(X)} flors -> coef_ = {model.coef_[0]}, "
      f"intercept_ = {model.intercept_[0]:.4f}")
print(f"Entrenament retallat: {len(X_retallat)} flors -> coef_ = "
      f"{model_retallat.coef_[0]}, intercept_ = {model_retallat.intercept_[0]:.4f}")
print(f"Frontera idèntica: {np.allclose(model.coef_, model_retallat.coef_)}")

Hem tret 40 de les 100 flors i la frontera **no ha canviat ni un
decimal**. L'SVM no necessita totes les dades per decidir; només
necessita els punts que estan al límit.

## 4. Quan cap recta serveix: el truc del kernel

Fins ara les dues classes es podien separar amb una recta. Però no
sempre és així. Fabriquem un problema on és impossible: un cercle de
punts dins d'un altre cercle de punts.

In [ ]:
from sklearn.datasets import make_circles
from sklearn.model_selection import train_test_split

Xc, yc = make_circles(n_samples=300, noise=0.08, factor=0.4, random_state=42)
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(
    Xc, yc, test_size=0.3, random_state=42, stratify=yc)

plt.figure(figsize=(6, 6))
plt.scatter(Xc[yc == 0, 0], Xc[yc == 0, 1], label="classe 0")
plt.scatter(Xc[yc == 1, 0], Xc[yc == 1, 1], label="classe 1")
plt.title("Un cercle dins d'un altre: cap recta separa això")
plt.legend()
plt.axis("equal")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
model_lineal = SVC(kernel="linear", random_state=42)
model_lineal.fit(Xc_tr, yc_tr)
print(f"SVM lineal sobre els cercles: {model_lineal.score(Xc_te, yc_te):.1%} de precisió")

Pràcticament tira una moneda a l'aire. Cap recta pot separar un cercle
de dins d'un cercle de fora.

**La idea**: si no hi ha una recta al pla $(x_1, x_2)$, potser sí que
n'hi ha una en un espai amb una dimensió més. Afegim una tercera
coordenada calculada a partir de les dues que ja teníem:

$$x_3 = x_1^2 + x_2^2$$

Això és, ni més ni menys, la distància al centre (al quadrat). Els
punts del cercle interior tindran un $x_3$ petit; els de l'exterior, un
$x_3$ gran.

In [ ]:
x3 = Xc[:, 0]**2 + Xc[:, 1]**2

fig = plt.figure(figsize=(8, 7))
ax = fig.add_subplot(projection="3d")
ax.scatter(Xc[yc == 0, 0], Xc[yc == 0, 1], x3[yc == 0], label="classe 0")
ax.scatter(Xc[yc == 1, 0], Xc[yc == 1, 1], x3[yc == 1], label="classe 1")
ax.set_xlabel("x1")
ax.set_ylabel("x2")
ax.set_zlabel("x3 = x1² + x2²")
ax.set_title("La mateixa dada, amb una dimensió afegida")
ax.legend()
plt.show()

En 3D, els dos grups queden a alçades ($x_3$) diferents: **un pla
horitzontal els separa sense problemes**. El que en 2D era impossible
amb una recta, en 3D és trivial amb un pla.

Aquest és exactament el truc del kernel: **buscar una separació lineal
en un espai amb més dimensions**. La diferència és que l'SVM amb un
`kernel="rbf"` no calcula $x_3$ de veritat ni cap altra dimensió nova
explícitament — fa servir un truc matemàtic (la "funció nucli") que
dona el mateix resultat que si les hagués calculat, sense pagar el cost
de construir-les. Per a nosaltres, el que importa és el resultat:

In [ ]:
model_rbf = SVC(kernel="rbf", random_state=42)
model_rbf.fit(Xc_tr, yc_tr)
print(f"SVM amb kernel rbf sobre els cercles: {model_rbf.score(Xc_te, yc_te):.1%} de precisió")

D'un cop de gambada de moneda a l'aire a encertar-ho (gairebé) tot,
només canviant una paraula: `kernel="rbf"`.

## 5. Els dos paràmetres que importen: C i gamma

- **`C`**: com de dur és el model amb els errors d'entrenament. Un `C`
  alt no perdona cap error i pot sobreajustar; un `C` baix deixa passar
  algun error a canvi d'una frontera més suau.
- **`gamma`** (només amb `kernel="rbf"`): com de "local" és la
  influència de cada punt. Un `gamma` alt fa que cada punt només
  influeixi el seu entorn immediat; la frontera s'arruga al voltant de
  cada mostra. Un `gamma` baix fa una frontera molt més suau.

Fem servir uns cercles amb una mica més de soroll que els del punt
anterior: amb dades massa netes, qualsevol `gamma` encerta el 100 % i
no es veu l'efecte de sobreajustar.

In [ ]:
Xg, yg = make_circles(n_samples=200, noise=0.3, factor=0.4, random_state=42)
Xg_tr, Xg_te, yg_tr, yg_te = train_test_split(
    Xg, yg, test_size=0.3, random_state=42, stratify=yg)

def dibuixa_regio(ax, model, X, y, titol):
    x1_min, x1_max = X[:, 0].min() - 0.3, X[:, 0].max() + 0.3
    x2_min, x2_max = X[:, 1].min() - 0.3, X[:, 1].max() + 0.3
    xx, yy = np.meshgrid(np.linspace(x1_min, x1_max, 200),
                          np.linspace(x2_min, x2_max, 200))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.25, cmap="coolwarm")
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", edgecolors="k", s=15)
    ax.set_title(titol, fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])

valors_C = [0.1, 1, 100]
valors_gamma = [0.1, 1, 50]

fig, axs = plt.subplots(len(valors_C), len(valors_gamma), figsize=(11, 11))
resultats_grid = []
for i, C in enumerate(valors_C):
    for j, gamma in enumerate(valors_gamma):
        m = SVC(kernel="rbf", C=C, gamma=gamma, random_state=42)
        m.fit(Xg_tr, yg_tr)
        train_acc = m.score(Xg_tr, yg_tr)
        test_acc = m.score(Xg_te, yg_te)
        resultats_grid.append({"C": C, "gamma": gamma,
                                "precisió entrenament": train_acc,
                                "precisió test": test_acc})
        dibuixa_regio(axs[i, j], m, Xg_tr, yg_tr,
                      f"C={C}, gamma={gamma}\nentren.: {train_acc:.0%}  test: {test_acc:.0%}")
plt.tight_layout()
plt.show()

pd.DataFrame(resultats_grid)

Fixa't en la cantonada `C=100, gamma=50`: **100 % a l'entrenament**,
però la precisió de test cau fins al 76,7 %. És la separació més gran
entre les dues (23 punts) de tota la graella: la frontera s'ha tancat
tant al voltant de cada punt d'entrenament que ha memoritzat el soroll
en comptes d'aprendre la forma del cercle. Això és **sobreajustar**:
un 100 % a l'entrenament que no es tradueix en un bon resultat al test.

Amb `gamma` massa baix i `C` massa baix (`C=0.1, gamma=0.1`), passa el
contrari: ni l'entrenament (67,9 %) ni el test (61,7 %) surten bé. La
frontera és gairebé plana i no capta la forma circular en cap dels dos
conjunts; això és **infraajustar**. `C` mou el mateix equilibri des
d'un altre angle: com de disposat està el model a acceptar algun error
a canvi d'una frontera més simple.

## 6. Per què l'escalat és obligatori aquí

L'SVM, igual que k-NN, treballa amb distàncies (la fórmula del marge
del punt 2 té una arrel quadrada de sumes de quadrats: és Pitàgoras).
Si una columna té valors molt més grans que les altres, dominarà la
distància encara que no sigui la característica més important.

Ho comprovem amb les 4 columnes d'Iris i les 3 espècies.

In [ ]:
dades = dades.rename(columns={
    "sepal length (cm)": "sepal_llarg",
    "sepal width (cm)": "sepal_ample",
})
X_complet = dades[["sepal_llarg", "sepal_ample", "petal_llarg", "petal_ample"]].copy()
y_complet = dades["especie"]

Xc_tr2, Xc_te2, yc_tr2, yc_te2 = train_test_split(
    X_complet, y_complet, test_size=0.3, random_state=42, stratify=y_complet)

model_normal = SVC(kernel="rbf", random_state=42)
model_normal.fit(Xc_tr2, yc_tr2)
print(f"Precisió amb les dades tal com vénen: {model_normal.score(Xc_te2, yc_te2):.1%}")

In [ ]:
# Multipliquem una sola columna per 100 (com si l'haguéssim mesurat en
# una altra unitat). Les distàncies queden dominades per aquesta columna.
Xc_tr3 = Xc_tr2.copy()
Xc_te3 = Xc_te2.copy()
Xc_tr3["sepal_llarg"] *= 100
Xc_te3["sepal_llarg"] *= 100

model_sense_escalar = SVC(kernel="rbf", random_state=42)
model_sense_escalar.fit(Xc_tr3, yc_tr2)
print(f"Precisió amb una columna x100, sense escalar: "
      f"{model_sense_escalar.score(Xc_te3, yc_te2):.1%}")

La precisió cau en picat només per haver canviat la unitat d'una
columna. El model no sap que "100x" no és informació de veritat.

La solució és estandarditzar totes les columnes abans d'entrenar, dins
d'un `Pipeline` perquè l'escalat s'aprengui només amb les dades
d'entrenament.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

pipeline = make_pipeline(StandardScaler(), SVC(kernel="rbf", random_state=42))
pipeline.fit(Xc_tr3, yc_tr2)
print(f"Precisió amb una columna x100 + StandardScaler dins d'un Pipeline: "
      f"{pipeline.score(Xc_te3, yc_te2):.1%}")

Amb l'`StandardScaler` dins del `Pipeline`, la unitat de mesura ja no
importa: totes les columnes queden a la mateixa escala abans que l'SVM
les vegi, i la precisió torna a ser pràcticament la mateixa que amb les
dades originals.

## 7. Pràctica

### Exercici 1 — Calcula un marge

Tria els teus propis $w_1, w_2, b$ per a una recta que separi `X` i `y`
del punt 1 (pots inspirar-te en les rectes A, B o C, o inventar-te'n
una altra). Calcula'n el marge amb la funció `distancia` del punt 2 i
compara'l amb el de la Recta C.

In [ ]:
# La teva recta: w1*x1 + w2*x2 + b = 0
w1, w2, b_propi = None, None, None  # substitueix pels teus valors

# Calcula el marge amb la funció `distancia` i compara'l amb el 0.636 de la Recta C

### Exercici 2 — Quants vectors de suport?

Entrena un `SVC(kernel="linear")` sobre les 4 columnes i les 3 espècies
d'Iris (`X_complet`, `y_complet`). Compta quants vectors de suport té
(`model.support_vectors_`) i quin percentatge representen sobre el
total de mostres d'entrenament.

In [ ]:
# model = SVC(kernel="linear", random_state=42)
# model.fit(...)
# nombre de vectors de suport i percentatge sobre el total

### Exercici 3 — Compara kernels

Sobre el problema dels cercles (`Xc_tr`, `yc_tr`, `Xc_te`, `yc_te`),
entrena tres SVM amb `kernel="linear"`, `kernel="poly"` i
`kernel="rbf"`, i compara les tres precisions en una taula.

In [ ]:
# for kernel in ["linear", "poly", "rbf"]:
#     entrena un SVC amb aquest kernel i guarda la precisió de test

### Exercici 4 — La millor combinació de C i gamma

Fes servir `GridSearchCV` per buscar la millor combinació de `C` i
`gamma` (`kernel="rbf"`) sobre el problema dels cercles. Prova almenys
4 valors de cada paràmetre. Quina combinació guanya i quina precisió
té?

In [ ]:
# from sklearn.model_selection import GridSearchCV
# graella = {"C": [...], "gamma": [...]}
# cerca = GridSearchCV(SVC(kernel="rbf"), graella, cv=5)
# cerca.fit(Xc_tr, yc_tr)
# millors parametres i precisio al test

## 8. Resum

- Davant de moltes rectes que separen bé les dades, l'SVM tria **la de
  marge màxim**: la que queda més lluny de tots els punts.
- El marge es calcula amb geometria de batxillerat (distància
  punt-recta, Pitàgoras).
- **Només els vectors de suport determinen la frontera**; la resta de
  punts podrien desaparèixer i no canviaria res.
- Quan cap recta serveix, el **kernel** busca una separació lineal en
  un espai amb més dimensions, sense arribar a construir-les de
  veritat.
- `C` i `gamma` controlen com de dura és la frontera amb els errors i
  com de local és la seva forma; massa `gamma` sobreajusta.
- L'SVM treballa amb distàncies: **l'escalat no és opcional**.

### El tancament de la sèrie

Cinc models, cinc maneres diferents de mirar les mateixes dades:

| Model | Com decideix | Escalat necessari | Es pot interpretar | Quan convé |
|---|---|---|---|---|
| **k-NN** | Mira les $k$ mostres més properes i vota | Sí (treballa amb distàncies) | Poc: no hi ha regles, només veïns | Poques dades, fronteres irregulars, quan no cal explicar la decisió |
| **Arbre de decisió** | Cadena de condicions `if/else` trobades sol | No | Molt: es pot dibuixar i llegir | Quan cal explicar la decisió a algú, dades amb relacions no lineals |
| **Bosc aleatori** | Cent arbres diferents voten i es queda la majoria | No | Mitjana: es pot veure la importància de cada columna, no cada regla | Quan un sol arbre sobreajusta i cal robustesa, sense preocupar-se de l'escalat |
| **Regressió logística** | Una recta (o pla) amb una probabilitat associada | Sí (els coeficients depenen de l'escala) | Molt: cada coeficient diu el pes d'una columna | Problemes linealment separables on cal saber "quant seguim de segurs" |
| **SVM** | La recta (o frontera) que deixa el marge més gran | Sí (treballa amb distàncies) | Poc amb kernel no lineal; alta amb kernel lineal i poques columnes | Fronteres clares amb marge, o problemes no lineals amb kernel rbf, en conjunts no gegantins |

La pregunta no és "quin model és el millor", és **quin encaixa amb les
teves dades**: si necessites explicar la decisió, si les dades vénen a
escales molt diferents, si la frontera és una recta o té una forma
estranya, i quantes dades tens per entrenar.